# Experiment 13: CatBoost + Stacking + Optuna Tuning 🏆

## Breaking Through 90% Weighted F1

Experiment 12 reached **89.8% weighted F1** (5-fold CV) — just 0.2pp shy of 90%. This experiment adds three key improvements:

### What's New vs Experiment 12:

| Improvement | Why It Helps |
|-------------|-------------|
| **CatBoost** (4th model) | Ordered boosting adds diversity — disagrees with LGB/XGB on edge cases |
| **Optuna tuning** for LGB & XGB | Hand-picked params → Bayesian-optimized params (squeeze 0.1-0.3pp each) |
| **Stacking meta-learner** | Logistic regression learns *class-specific* model weighting instead of fixed weights |

### Pipeline:
1. **Optuna** tunes LightGBM & XGBoost params (3-fold inner CV, 50 trials each)
2. **5-Fold CV** (exact MAPS protocol, `StratifiedKFold(5, seed=7325111)`):
   - Train MLP, LightGBM (tuned), XGBoost (tuned), CatBoost per fold
   - Train stacking meta-learner on validation-set predictions
   - Predict on test fold → pool all predictions
3. Compare: individual models, simple average, stacking ensemble

---
**Runtime**: ~90-120 min on Kaggle P100 (100 Optuna trials + 5 folds × 4 models)

In [ ]:
import os
import sys
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, SequentialSampler

from sklearn.metrics import (
    f1_score, accuracy_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
import optuna
from optuna.samplers import TPESampler

# Suppress noisy logs
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"Optuna: {optuna.__version__}")
print(f"Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")

## 1. Load and Prepare Data

In [ ]:
# Load data
df = pd.read_csv("/kaggle/input/datasets/amshahriarrashidmahe/chl-codex-annotated/cHL_CODEX_annotation.csv")

MARKER_COLS = [
    'BCL.2', 'CCR6', 'CD11b', 'CD11c', 'CD15', 'CD16', 'CD162', 'CD163',
    'CD2', 'CD20', 'CD206', 'CD25', 'CD30', 'CD31', 'CD4', 'CD44',
    'CD45', 'CD45RA', 'CD45RO', 'CD5', 'CD56', 'CD57', 'CD68', 'CD69',
    'CD7', 'CD8', 'Collagen.4', 'Cytokeratin', 'DAPI.01', 'EGFR',
    'FoxP3', 'Granzyme.B', 'HLA.DR', 'IDO.1', 'LAG.3', 'MCT', 'MMP.9',
    'MUC.1', 'PD.1', 'PD.L1', 'Podoplanin', 'T.bet', 'TCR.g.d', 'TCRb',
    'Tim.3', 'VISA', 'Vimentin', 'a.SMA', 'b.Catenin'
]
FEATURE_COLS = MARKER_COLS + ['cellSize']  # 50 features

CLASS_NAMES = ['B', 'CD4', 'CD8', 'DC', 'Endothelial', 'Epithelial',
               'Lymphatic', 'M1', 'M2', 'Mast', 'Monocyte', 'NK',
               'Neutrophil', 'Other', 'TReg', 'Tumor']
LABEL_MAP = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
NUM_FEATURES = len(FEATURE_COLS)

# Filter and extract
df = df[df['cellType'].isin(CLASS_NAMES)].reset_index(drop=True)
X_all = df[FEATURE_COLS].values.astype(np.float64)
y_all = df['cellType'].map(LABEL_MAP).values.astype(np.int64)

print(f"Dataset: {len(df):,} cells | Features: {NUM_FEATURES} | Classes: {NUM_CLASSES}")
print(f"\nClass distribution:")
for name, idx in sorted(LABEL_MAP.items(), key=lambda x: x[1]):
    count = (y_all == idx).sum()
    print(f"  {name:15s}: {count:6,} ({count/len(y_all)*100:5.1f}%)")

## 2. MLP Infrastructure (same as Experiments 10-12)

In [ ]:
# ====================================================================
# MLP INFRASTRUCTURE
# ====================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 7325111
BATCH_SIZE = 128


def set_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class CellDataset(Dataset):
    def __init__(self, X, y, is_train=True, mean=None, std=None):
        self.y = y
        if is_train:
            self.mean = np.mean(X, axis=0)
            self.std = np.std(X, axis=0)
        else:
            self.mean, self.std = mean, std
        self.x = (X - self.mean) / (self.std + 1e-12)

    def __len__(self): return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx] / 255.0, int(self.y[idx])


class MLP(nn.Module):
    def __init__(self, input_dim=50, hidden_dim=512, num_classes=16, dropout=0.10):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Dropout(p=dropout)
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, batch):
        features = self.fc(batch)
        logits = self.classifier(features)
        probs = torch.softmax(logits, dim=-1)
        return logits, probs


def train_mlp_fold(X_train, y_train, X_valid, y_valid, X_test, y_test):
    """Train MLP on one fold, return test probabilities."""
    set_seed(SEED)

    ds_tr = CellDataset(X_train, y_train, is_train=True)
    ds_val = CellDataset(X_valid, y_valid, is_train=False,
                         mean=ds_tr.mean, std=ds_tr.std)
    ds_te = CellDataset(X_test, y_test, is_train=False,
                        mean=ds_tr.mean, std=ds_tr.std)

    # Weighted sampler
    ll = y_train.tolist()
    n_s = float(len(ll))
    ul = sorted(set(ll))
    wpc = {c: n_s / ll.count(c) for c in ul}
    sw = [wpc[l] for l in ll]

    tr_dl = DataLoader(ds_tr, batch_size=BATCH_SIZE,
                       sampler=WeightedRandomSampler(sw, len(sw)),
                       drop_last=True, num_workers=2)
    val_dl = DataLoader(ds_val, batch_size=BATCH_SIZE,
                        sampler=SequentialSampler(ds_val), drop_last=False, num_workers=2)
    te_dl = DataLoader(ds_te, batch_size=BATCH_SIZE,
                       sampler=SequentialSampler(ds_te), drop_last=False, num_workers=2)

    model = MLP(input_dim=NUM_FEATURES, hidden_dim=512,
                num_classes=NUM_CLASSES, dropout=0.10)
    model.to(DEVICE, dtype=torch.float64)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    best_vloss = float('inf')
    pat = 0
    best_st = None

    for epoch in range(500):
        model.train()
        for feat, lab in tr_dl:
            feat, lab = feat.to(DEVICE), lab.to(DEVICE)
            logits, _ = model(feat)
            optimizer.zero_grad()
            loss_fn(logits, lab).backward()
            optimizer.step()

        model.eval()
        vl = 0
        with torch.no_grad():
            for feat, lab in val_dl:
                feat, lab = feat.to(DEVICE), lab.to(DEVICE)
                logits, _ = model(feat)
                vl += loss_fn(logits, lab).item()
        vl /= len(val_dl)

        if vl < best_vloss:
            best_vloss = vl
            best_st = {k: v.clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1

        if pat > 150 and epoch >= 350:
            break

    # Get test + valid probabilities
    model.load_state_dict(best_st)
    model.eval()

    test_probs = []
    with torch.no_grad():
        for feat, _ in te_dl:
            feat = feat.to(DEVICE)
            _, probs = model(feat)
            test_probs.append(probs.cpu().numpy())

    valid_probs = []
    with torch.no_grad():
        for feat, _ in val_dl:
            feat = feat.to(DEVICE)
            _, probs = model(feat)
            valid_probs.append(probs.cpu().numpy())

    return np.concatenate(test_probs, axis=0), np.concatenate(valid_probs, axis=0)


print("✅ MLP infrastructure defined")

## 3. Optuna Hyperparameter Tuning

Tune LightGBM and XGBoost using **Bayesian optimization** (TPE sampler).

- **Inner evaluation**: 3-fold stratified CV on the full dataset
- **Objective**: Maximize weighted F1 score
- **Trials**: 100 each (~30-40 min total on GPU)
- The best params are then used in the main 5-fold evaluation

In [ ]:
# ====================================================================
# OPTUNA TUNING — LightGBM
# ====================================================================
# Pre-scale data for tuning (z-score)
scaler_tune = StandardScaler()
X_all_scaled = scaler_tune.fit_transform(X_all)

# Inner CV for tuning
tune_skf = StratifiedKFold(n_splits=3, random_state=42, shuffle=True)


def lgb_objective(trial):
    """Optuna objective for LightGBM — maximize weighted F1."""
    params = {
        'objective': 'multiclass',
        'num_class': NUM_CLASSES,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': SEED,
        'n_jobs': -1,
        'is_unbalance': True,
        # Tuned hyperparameters
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 63, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
    }

    fold_scores = []
    for tr_idx, te_idx in tune_skf.split(X_all_scaled, y_all):
        lgb_tr = lgb.Dataset(X_all_scaled[tr_idx], label=y_all[tr_idx])
        lgb_val = lgb.Dataset(X_all_scaled[te_idx], label=y_all[te_idx], reference=lgb_tr)

        model = lgb.train(
            params, lgb_tr, num_boost_round=500,
            valid_sets=[lgb_val], valid_names=['valid'],
            callbacks=[lgb.early_stopping(stopping_rounds=30),
                       lgb.log_evaluation(period=0)]
        )
        preds = np.argmax(model.predict(X_all_scaled[te_idx]), axis=1)
        fold_scores.append(f1_score(y_all[te_idx], preds, average='weighted'))

    return np.mean(fold_scores)


print("🔍 Tuning LightGBM with Optuna (100 trials, 3-fold CV)...")
t0 = time.time()

lgb_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    study_name='lgb_tuning'
)
lgb_study.optimize(lgb_objective, n_trials=100, show_progress_bar=True)

lgb_best_params = lgb_study.best_params
lgb_best_score = lgb_study.best_value

print(f"\n✅ LightGBM tuning done in {time.time()-t0:.0f}s")
print(f"   Best 3-fold CV W-F1: {lgb_best_score:.4f}")
print(f"   Best params: {lgb_best_params}")

In [ ]:
# ====================================================================
# OPTUNA TUNING — XGBoost
# ====================================================================
def xgb_objective(trial):
    """Optuna objective for XGBoost — maximize weighted F1."""
    params = {
        'objective': 'multi:softprob',
        'num_class': NUM_CLASSES,
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed': SEED,
        'verbosity': 0,
        # Tuned hyperparameters
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-4, 5.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }

    fold_scores = []
    for tr_idx, te_idx in tune_skf.split(X_all_scaled, y_all):
        # Sample weights for class imbalance
        cc = np.bincount(y_all[tr_idx])
        cwm = {i: len(y_all[tr_idx]) / (NUM_CLASSES * c) for i, c in enumerate(cc)}
        sw = np.array([cwm[l] for l in y_all[tr_idx]])

        dtrain = xgb.DMatrix(X_all_scaled[tr_idx], label=y_all[tr_idx],
                             weight=sw, feature_names=FEATURE_COLS)
        dvalid = xgb.DMatrix(X_all_scaled[te_idx], label=y_all[te_idx],
                             feature_names=FEATURE_COLS)

        model = xgb.train(
            params, dtrain, num_boost_round=500,
            evals=[(dvalid, 'valid')],
            early_stopping_rounds=30, verbose_eval=0
        )
        preds = np.argmax(model.predict(dvalid), axis=1)
        fold_scores.append(f1_score(y_all[te_idx], preds, average='weighted'))

    return np.mean(fold_scores)


print("🔍 Tuning XGBoost with Optuna (100 trials, 3-fold CV)...")
t0 = time.time()

xgb_study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    study_name='xgb_tuning'
)
xgb_study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

xgb_best_params = xgb_study.best_params
xgb_best_score = xgb_study.best_value

print(f"\n✅ XGBoost tuning done in {time.time()-t0:.0f}s")
print(f"   Best 3-fold CV W-F1: {xgb_best_score:.4f}")
print(f"   Best params: {xgb_best_params}")

In [ ]:
# ====================================================================
# TUNING SUMMARY — Compare Exp 12 defaults vs Optuna-tuned
# ====================================================================
print("=" * 70)
print("  HYPERPARAMETER TUNING SUMMARY")
print("=" * 70)

print("\n  LightGBM:")
print(f"    Exp 12 default  → 3-fold CV W-F1: (baseline ~89.3%)")
print(f"    Optuna-tuned    → 3-fold CV W-F1: {lgb_best_score:.4f}")
print(f"    Best params:")
for k, v in lgb_best_params.items():
    print(f"      {k}: {v}")

print(f"\n  XGBoost:")
print(f"    Exp 12 default  → 3-fold CV W-F1: (baseline ~89.2%)")
print(f"    Optuna-tuned    → 3-fold CV W-F1: {xgb_best_score:.4f}")
print(f"    Best params:")
for k, v in xgb_best_params.items():
    print(f"      {k}: {v}")
print("=" * 70)

---
## 4. 5-Fold Cross-Validation with Stacking

### Exact MAPS protocol (`StratifiedKFold(5, seed=7325111)`)

For **each fold**:
1. **Outer split**: 80% train_pool / 20% test
2. **Inner split**: train_pool → 80% train / 20% valid (for early stopping)
3. **Train 4 models**: MLP, LightGBM (Optuna), XGBoost (Optuna), CatBoost
4. **Stacking**: Get all 4 models' predictions on validation set → train a Logistic Regression meta-learner → predict on test set
5. **Pool** all test predictions across 5 folds → final metrics on all 143K cells

In [ ]:
# ====================================================================
# 5-FOLD CV: MLP + LightGBM (tuned) + XGBoost (tuned) + CatBoost + STACKING
# ====================================================================
OUTER_SEED = 7325111  # Exact MAPS seed

outer_skf = StratifiedKFold(n_splits=5, random_state=OUTER_SEED, shuffle=True)

# Storage for pooled predictions
pooled_labels = []
pooled_mlp_preds = []
pooled_lgb_preds = []
pooled_xgb_preds = []
pooled_cat_preds = []
pooled_stack_preds = []
pooled_avg4_preds = []

pooled_mlp_probs = []
pooled_lgb_probs = []
pooled_xgb_probs = []
pooled_cat_probs = []
pooled_stack_probs = []

fold_results_cv = []

total_t0 = time.time()

for fold_idx, (train_pool_idx, test_idx) in enumerate(outer_skf.split(X_all, y_all)):
    fold_t0 = time.time()
    print(f"\n{'='*80}")
    print(f"  FOLD {fold_idx} — Train pool: {len(train_pool_idx):,} | Test: {len(test_idx):,}")
    print(f"{'='*80}")

    # Inner split for early stopping (same as MAPS)
    inner_skf = StratifiedKFold(n_splits=5, random_state=OUTER_SEED, shuffle=True)
    for train_idx, valid_idx in inner_skf.split(X_all[train_pool_idx], y_all[train_pool_idx]):
        train_idx = train_pool_idx[train_idx]
        valid_idx = train_pool_idx[valid_idx]
        break  # Only first inner split

    X_fold_train = X_all[train_idx]
    y_fold_train = y_all[train_idx]
    X_fold_valid = X_all[valid_idx]
    y_fold_valid = y_all[valid_idx]
    X_fold_test = X_all[test_idx]
    y_fold_test = y_all[test_idx]

    print(f"  Split: Train={len(train_idx):,} | Valid={len(valid_idx):,} | Test={len(test_idx):,}")

    # Scale data for tree models
    fold_scaler = StandardScaler()
    X_tr_sc = fold_scaler.fit_transform(X_fold_train)
    X_val_sc = fold_scaler.transform(X_fold_valid)
    X_te_sc = fold_scaler.transform(X_fold_test)

    # ================================================================
    # 1. TRAIN MLP
    # ================================================================
    print(f"  [1/4] Training MLP...")
    mlp_te_probs, mlp_val_probs = train_mlp_fold(
        X_fold_train, y_fold_train,
        X_fold_valid, y_fold_valid,
        X_fold_test, y_fold_test
    )
    mlp_te_pred = np.argmax(mlp_te_probs, axis=1)
    print(f"         MLP test W-F1: {f1_score(y_fold_test, mlp_te_pred, average='weighted'):.4f}")

    # ================================================================
    # 2. TRAIN LightGBM (Optuna-tuned)
    # ================================================================
    print(f"  [2/4] Training LightGBM (Optuna-tuned)...")
    lgb_params_tuned = {
        'objective': 'multiclass',
        'num_class': NUM_CLASSES,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': SEED,
        'n_jobs': -1,
        'is_unbalance': True,
        **lgb_best_params  # Inject Optuna-tuned params
    }

    lgb_tr = lgb.Dataset(X_tr_sc, label=y_fold_train, feature_name=FEATURE_COLS)
    lgb_val = lgb.Dataset(X_val_sc, label=y_fold_valid, feature_name=FEATURE_COLS, reference=lgb_tr)

    lgb_m = lgb.train(
        lgb_params_tuned, lgb_tr, num_boost_round=1000,
        valid_sets=[lgb_val], valid_names=['valid'],
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=0)]
    )

    lgb_te_probs = lgb_m.predict(X_te_sc)
    lgb_val_probs = lgb_m.predict(X_val_sc)
    lgb_te_pred = np.argmax(lgb_te_probs, axis=1)
    print(f"         LGB test W-F1: {f1_score(y_fold_test, lgb_te_pred, average='weighted'):.4f}")

    # ================================================================
    # 3. TRAIN XGBoost (Optuna-tuned)
    # ================================================================
    print(f"  [3/4] Training XGBoost (Optuna-tuned)...")
    xgb_params_tuned = {
        'objective': 'multi:softprob',
        'num_class': NUM_CLASSES,
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'seed': SEED,
        'verbosity': 0,
        **xgb_best_params  # Inject Optuna-tuned params
    }

    # Sample weights for class imbalance
    cc = np.bincount(y_fold_train)
    cwm = {i: len(y_fold_train) / (NUM_CLASSES * c) for i, c in enumerate(cc)}
    sw_xgb = np.array([cwm[l] for l in y_fold_train])

    dt = xgb.DMatrix(X_tr_sc, label=y_fold_train, weight=sw_xgb, feature_names=FEATURE_COLS)
    dv = xgb.DMatrix(X_val_sc, label=y_fold_valid, feature_names=FEATURE_COLS)
    dte = xgb.DMatrix(X_te_sc, label=y_fold_test, feature_names=FEATURE_COLS)

    xgb_m = xgb.train(
        xgb_params_tuned, dt, num_boost_round=1000,
        evals=[(dv, 'valid')], early_stopping_rounds=50, verbose_eval=0
    )

    xgb_te_probs = xgb_m.predict(dte)
    xgb_val_probs = xgb_m.predict(dv)
    xgb_te_pred = np.argmax(xgb_te_probs, axis=1)
    print(f"         XGB test W-F1: {f1_score(y_fold_test, xgb_te_pred, average='weighted'):.4f}")

    # ================================================================
    # 4. TRAIN CatBoost
    # ================================================================
    print(f"  [4/4] Training CatBoost...")
    cat_model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=SEED,
        task_type='GPU' if torch.cuda.is_available() else 'CPU',
        auto_class_weights='Balanced',
        eval_metric='TotalF1:average=Weighted',
        early_stopping_rounds=50,
        verbose=0,
    )

    cat_model.fit(
        X_tr_sc, y_fold_train,
        eval_set=(X_val_sc, y_fold_valid),
        use_best_model=True
    )

    cat_te_probs = cat_model.predict_proba(X_te_sc)
    cat_val_probs = cat_model.predict_proba(X_val_sc)
    cat_te_pred = np.argmax(cat_te_probs, axis=1)
    print(f"         CAT test W-F1: {f1_score(y_fold_test, cat_te_pred, average='weighted'):.4f}")

    # ================================================================
    # 5. STACKING META-LEARNER
    # ================================================================
    print(f"  [Stack] Training Logistic Regression meta-learner...")

    # Build meta-features from validation predictions (4 models × 16 classes = 64 features)
    meta_valid = np.hstack([mlp_val_probs, lgb_val_probs, xgb_val_probs, cat_val_probs])
    meta_test = np.hstack([mlp_te_probs, lgb_te_probs, xgb_te_probs, cat_te_probs])

    # Train logistic regression on validation meta-features
    stack_model = LogisticRegression(
        max_iter=1000,
        multi_class='multinomial',
        solver='lbfgs',
        C=1.0,
        random_state=SEED
    )
    stack_model.fit(meta_valid, y_fold_valid)

    stack_te_probs = stack_model.predict_proba(meta_test)
    stack_te_pred = np.argmax(stack_te_probs, axis=1)

    # Simple average of all 4 models (baseline comparison)
    avg4_te_probs = (mlp_te_probs + lgb_te_probs + xgb_te_probs + cat_te_probs) / 4.0
    avg4_te_pred = np.argmax(avg4_te_probs, axis=1)

    # ================================================================
    # FOLD SUMMARY
    # ================================================================
    fold_res = {
        'fold': fold_idx,
        'mlp_wf1': f1_score(y_fold_test, mlp_te_pred, average='weighted'),
        'lgb_wf1': f1_score(y_fold_test, lgb_te_pred, average='weighted'),
        'xgb_wf1': f1_score(y_fold_test, xgb_te_pred, average='weighted'),
        'cat_wf1': f1_score(y_fold_test, cat_te_pred, average='weighted'),
        'avg4_wf1': f1_score(y_fold_test, avg4_te_pred, average='weighted'),
        'stack_wf1': f1_score(y_fold_test, stack_te_pred, average='weighted'),
    }
    fold_results_cv.append(fold_res)

    print(f"\n  Fold {fold_idx} results:")
    print(f"    MLP:       {fold_res['mlp_wf1']:.4f}")
    print(f"    LightGBM:  {fold_res['lgb_wf1']:.4f}")
    print(f"    XGBoost:   {fold_res['xgb_wf1']:.4f}")
    print(f"    CatBoost:  {fold_res['cat_wf1']:.4f}")
    print(f"    Avg (4):   {fold_res['avg4_wf1']:.4f}")
    print(f"    STACKING:  {fold_res['stack_wf1']:.4f} {'🎯' if fold_res['stack_wf1'] >= 0.90 else ''}")
    print(f"    (fold time: {time.time()-fold_t0:.0f}s)")

    # Pool predictions
    pooled_labels.extend(y_fold_test.tolist())
    pooled_mlp_preds.extend(mlp_te_pred.tolist())
    pooled_lgb_preds.extend(lgb_te_pred.tolist())
    pooled_xgb_preds.extend(xgb_te_pred.tolist())
    pooled_cat_preds.extend(cat_te_pred.tolist())
    pooled_stack_preds.extend(stack_te_pred.tolist())
    pooled_avg4_preds.extend(avg4_te_pred.tolist())

    pooled_mlp_probs.append(mlp_te_probs)
    pooled_lgb_probs.append(lgb_te_probs)
    pooled_xgb_probs.append(xgb_te_probs)
    pooled_cat_probs.append(cat_te_probs)
    pooled_stack_probs.append(stack_te_probs)

# Concatenate
pooled_mlp_probs = np.concatenate(pooled_mlp_probs, axis=0)
pooled_lgb_probs = np.concatenate(pooled_lgb_probs, axis=0)
pooled_xgb_probs = np.concatenate(pooled_xgb_probs, axis=0)
pooled_cat_probs = np.concatenate(pooled_cat_probs, axis=0)
pooled_stack_probs = np.concatenate(pooled_stack_probs, axis=0)

print(f"\n{'='*80}")
print(f"  All 5 folds complete! Total time: {time.time()-total_t0:.0f}s")
print(f"  Pooled: {len(pooled_labels):,} test predictions")
print(f"{'='*80}")

## 5. Pooled 5-Fold Results

In [ ]:
# ====================================================================
# POOLED 5-FOLD RESULTS
# ====================================================================
pooled_labels = np.array(pooled_labels)
pooled_mlp_preds = np.array(pooled_mlp_preds)
pooled_lgb_preds = np.array(pooled_lgb_preds)
pooled_xgb_preds = np.array(pooled_xgb_preds)
pooled_cat_preds = np.array(pooled_cat_preds)
pooled_stack_preds = np.array(pooled_stack_preds)
pooled_avg4_preds = np.array(pooled_avg4_preds)

# Also compute tree-only ensembles from pooled probs
pooled_tree3_probs = (pooled_lgb_probs + pooled_xgb_probs + pooled_cat_probs) / 3.0
pooled_tree3_preds = np.argmax(pooled_tree3_probs, axis=1)

pooled_avg4_probs = (pooled_mlp_probs + pooled_lgb_probs + pooled_xgb_probs + pooled_cat_probs) / 4.0
pooled_avg4_preds_v2 = np.argmax(pooled_avg4_probs, axis=1)

results_5fold = {
    'MLP': pooled_mlp_preds,
    'LightGBM (tuned)': pooled_lgb_preds,
    'XGBoost (tuned)': pooled_xgb_preds,
    'CatBoost': pooled_cat_preds,
    'LGB+XGB+CAT Avg': pooled_tree3_preds,
    'Simple Avg (4 models)': pooled_avg4_preds_v2,
    'STACKING (LR meta)': pooled_stack_preds,
}

print("=" * 80)
print("   EXPERIMENT 13: 5-FOLD CV RESULTS (Pooled across ALL test folds)")
print("=" * 80)
print(f"   Total test predictions: {len(pooled_labels):,} cells")
print()
print(f"   {'Method':28s} {'W-F1':>8s} {'Micro-F1':>10s} {'Macro-F1':>10s} {'Acc':>8s} {'vs 90%':>8s}")
print(f"   {'─'*80}")

for name, preds in results_5fold.items():
    wf1 = f1_score(pooled_labels, preds, average='weighted')
    mif1 = f1_score(pooled_labels, preds, average='micro')
    maf1 = f1_score(pooled_labels, preds, average='macro')
    acc = accuracy_score(pooled_labels, preds)
    delta = f"{(wf1 - 0.90) * 100:+.1f}pp"
    marker = " 🎯" if wf1 >= 0.90 else ""
    print(f"   {name:28s} {wf1:8.4f} {mif1:10.4f} {maf1:10.4f} {acc:8.4f} {delta:>8s}{marker}")

print()

# Per-fold breakdown
print("   Per-Fold Weighted F1 Breakdown:")
print(f"   {'─'*90}")
print(f"   {'Fold':>6} │ {'MLP':>8} │ {'LGB':>8} │ {'XGB':>8} │ {'CAT':>8} │ {'Avg4':>8} │ {'Stack':>8}")
print(f"   {'─'*6}─┼─{'─'*8}─┼─{'─'*8}─┼─{'─'*8}─┼─{'─'*8}─┼─{'─'*8}─┼─{'─'*8}")
for r in fold_results_cv:
    print(f"   {r['fold']:>6} │ {r['mlp_wf1']:>8.4f} │ {r['lgb_wf1']:>8.4f} │ "
          f"{r['xgb_wf1']:>8.4f} │ {r['cat_wf1']:>8.4f} │ "
          f"{r['avg4_wf1']:>8.4f} │ {r['stack_wf1']:>8.4f}")

# Mean ± std
for key, label in [('mlp_wf1', 'MLP'), ('lgb_wf1', 'LightGBM'),
                    ('xgb_wf1', 'XGBoost'), ('cat_wf1', 'CatBoost'),
                    ('avg4_wf1', 'Avg (4)'), ('stack_wf1', 'Stacking')]:
    vals = [r[key] for r in fold_results_cv]
    print(f"   {label:12s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

print("=" * 80)

In [ ]:
# ====================================================================
# CLASSIFICATION REPORT — Stacking Ensemble
# ====================================================================
print("Classification Report — Stacking Ensemble (5-Fold Pooled):")
print("=" * 70)
print(classification_report(pooled_labels, pooled_stack_preds,
                            target_names=CLASS_NAMES, digits=4))

# Also show the best simple ensemble for comparison
print("\nClassification Report — Simple Average 4 Models (5-Fold Pooled):")
print("=" * 70)
print(classification_report(pooled_labels, pooled_avg4_preds_v2,
                            target_names=CLASS_NAMES, digits=4))

In [ ]:
# ====================================================================
# CONFUSION MATRICES — CatBoost vs Stacking Ensemble
# ====================================================================
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Left: CatBoost alone (the new model)
cm1 = confusion_matrix(pooled_labels, pooled_cat_preds)
cm1_norm = cm1.astype('float') / cm1.sum(axis=1, keepdims=True)
cat_wf1 = f1_score(pooled_labels, pooled_cat_preds, average='weighted')
sns.heatmap(cm1_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title(f'CatBoost 5-Fold (W-F1={cat_wf1:.4f})', fontsize=13)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)

# Right: Stacking Ensemble (the main result)
cm2 = confusion_matrix(pooled_labels, pooled_stack_preds)
cm2_norm = cm2.astype('float') / cm2.sum(axis=1, keepdims=True)
stack_wf1 = f1_score(pooled_labels, pooled_stack_preds, average='weighted')
sns.heatmap(cm2_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title(f'Stacking Ensemble 5-Fold (W-F1={stack_wf1:.4f})', fontsize=13)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('CatBoost vs Stacking Ensemble — Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# PER-FOLD BAR CHART — All 6 Methods
# ====================================================================
fig, ax = plt.subplots(figsize=(16, 8))

fold_nums = [r['fold'] for r in fold_results_cv]
x = np.arange(len(fold_nums))
width = 0.13

colors = ['#42A5F5', '#66BB6A', '#FFA726', '#AB47BC', '#78909C', '#FFD700']
labels_plot = ['MLP', 'LightGBM', 'XGBoost', 'CatBoost', 'Avg (4)', 'Stacking']
keys = ['mlp_wf1', 'lgb_wf1', 'xgb_wf1', 'cat_wf1', 'avg4_wf1', 'stack_wf1']

for i, (key, label, color) in enumerate(zip(keys, labels_plot, colors)):
    offset = (i - 2.5) * width
    vals = [r[key] * 100 for r in fold_results_cv]
    bars = ax.bar(x + offset, vals, width, label=label, color=color,
                  edgecolor='white' if i < 5 else 'black',
                  linewidth=1.5 if i == 5 else 0.5)

    # Annotate stacking bars
    if key == 'stack_wf1':
        for j, v in enumerate(vals):
            ax.text(x[j] + offset, v + 0.15, f"{v:.1f}%",
                    ha='center', va='bottom', fontweight='bold', fontsize=8, color='#B8860B')

ax.axhline(y=90, color='red', linestyle='--', alpha=0.7, linewidth=2, label='90% target')

ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i}' for i in fold_nums], fontsize=12)
ax.set_ylabel('Weighted F1 (%)', fontsize=12)
ax.set_title('Experiment 13: Per-Fold F1 — All Models + Stacking', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='lower right', ncol=2)
ax.set_ylim(82, 96)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# PER-CLASS F1 — Stacking vs Exp 12 Focus (which classes improved?)
# ====================================================================
fig, ax = plt.subplots(figsize=(16, 8))
x = np.arange(NUM_CLASSES)
width = 0.18

mlp_per = f1_score(pooled_labels, pooled_mlp_preds, average=None)
lgb_per = f1_score(pooled_labels, pooled_lgb_preds, average=None)
cat_per = f1_score(pooled_labels, pooled_cat_preds, average=None)
stack_per = f1_score(pooled_labels, pooled_stack_preds, average=None)

ax.bar(x - 1.5*width, mlp_per, width, label=f'MLP', alpha=0.8, color='#42A5F5')
ax.bar(x - 0.5*width, lgb_per, width, label=f'LightGBM (tuned)', alpha=0.8, color='#66BB6A')
ax.bar(x + 0.5*width, cat_per, width, label=f'CatBoost', alpha=0.8, color='#AB47BC')
ax.bar(x + 1.5*width, stack_per, width, label=f'Stacking Ensemble',
       alpha=0.9, color='gold', edgecolor='black')

ax.axhline(y=0.90, color='red', linestyle='--', alpha=0.7, label='90% target')

# Highlight weak classes
for i, name in enumerate(CLASS_NAMES):
    if stack_per[i] < 0.87:
        ax.annotate(f'{stack_per[i]:.2f}', (i + 1.5*width, stack_per[i]),
                    textcoords="offset points", xytext=(0, 8),
                    ha='center', fontsize=8, color='red', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: Individual Models vs Stacking Ensemble', fontsize=13)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# MODEL DIVERSITY ANALYSIS — How much do models disagree?
# ====================================================================
print("Model Agreement Analysis (5-fold pooled):")
print("=" * 60)

agree_all = ((pooled_mlp_preds == pooled_lgb_preds) &
             (pooled_lgb_preds == pooled_xgb_preds) &
             (pooled_xgb_preds == pooled_cat_preds))

agree_tree3 = ((pooled_lgb_preds == pooled_xgb_preds) &
               (pooled_xgb_preds == pooled_cat_preds))

pairs = [
    ('MLP-LGB', pooled_mlp_preds, pooled_lgb_preds),
    ('MLP-XGB', pooled_mlp_preds, pooled_xgb_preds),
    ('MLP-CAT', pooled_mlp_preds, pooled_cat_preds),
    ('LGB-XGB', pooled_lgb_preds, pooled_xgb_preds),
    ('LGB-CAT', pooled_lgb_preds, pooled_cat_preds),
    ('XGB-CAT', pooled_xgb_preds, pooled_cat_preds),
]

print(f"  All 4 agree:       {agree_all.mean():.1%} ({agree_all.sum():,} / {len(pooled_labels):,})")
print(f"  3 tree models agree: {agree_tree3.mean():.1%}")
print()
for name, p1, p2 in pairs:
    agree = (p1 == p2).mean()
    print(f"  {name:10s}: {agree:.1%}")

if agree_all.any():
    acc_agree = accuracy_score(pooled_labels[agree_all], pooled_mlp_preds[agree_all])
    acc_disagree = accuracy_score(pooled_labels[~agree_all], pooled_stack_preds[~agree_all])
    print(f"\n  When all 4 agree → accuracy:    {acc_agree:.4f}")
    print(f"  When they disagree → stack acc:  {acc_disagree:.4f}")
    print(f"  → Stacking resolves {(~agree_all).sum():,} disagreement cases")

In [ ]:
# ====================================================================
# FINAL COMPARISON: MAPS vs Exp 12 vs Exp 13
# ====================================================================
stack_wf1_final = f1_score(pooled_labels, pooled_stack_preds, average='weighted')
stack_mif1_final = f1_score(pooled_labels, pooled_stack_preds, average='micro')
avg4_wf1_final = f1_score(pooled_labels, pooled_avg4_preds_v2, average='weighted')
lgb_wf1_final = f1_score(pooled_labels, pooled_lgb_preds, average='weighted')
xgb_wf1_final = f1_score(pooled_labels, pooled_xgb_preds, average='weighted')
cat_wf1_final = f1_score(pooled_labels, pooled_cat_preds, average='weighted')
mlp_wf1_final = f1_score(pooled_labels, pooled_mlp_preds, average='weighted')

print("\n" + "=" * 70)
print("  EXPERIMENT 13 — FINAL COMPARISON")
print("=" * 70)

print(f"""
  ┌───────────────────────────────────────────────────────────────┐
  │  MAPS (Official)                                              │
  │    Single model:          86.8% weighted F1                   │
  │    5-fold ensemble:       89.7% weighted F1 / ~90% micro F1   │
  ├───────────────────────────────────────────────────────────────┤
  │  Experiment 12 — 5-Fold CV (default params, 3 models)         │
  │    Weighted Ens (MLP+LGB+XGB):  89.8% weighted F1             │
  ├───────────────────────────────────────────────────────────────┤
  │  Experiment 13 — 5-Fold CV (Optuna + CatBoost + Stacking)     │
  │    MLP (5-fold):          {mlp_wf1_final*100:.1f}% weighted F1                   │
  │    LightGBM (tuned):      {lgb_wf1_final*100:.1f}% weighted F1                   │
  │    XGBoost (tuned):       {xgb_wf1_final*100:.1f}% weighted F1                   │
  │    CatBoost:              {cat_wf1_final*100:.1f}% weighted F1                   │
  │    Simple Avg (4 models): {avg4_wf1_final*100:.1f}% weighted F1                   │
  │    STACKING ENSEMBLE:     {stack_wf1_final*100:.1f}% weighted F1 {"🎯" if stack_wf1_final >= 0.90 else ""}                │
  │    Stacking micro F1:     {stack_mif1_final*100:.1f}%                               │
  └───────────────────────────────────────────────────────────────┘
""")

if stack_wf1_final >= 0.90:
    improvement_vs_maps = (stack_wf1_final - 0.897) * 100
    improvement_vs_exp12 = (stack_wf1_final - 0.898) * 100
    print(f"  🎉 TARGET ACHIEVED: {stack_wf1_final*100:.1f}% WEIGHTED F1!")
    print(f"     vs MAPS 5-fold (89.7%):  +{improvement_vs_maps:.1f}pp")
    print(f"     vs Exp 12 (89.8%):       +{improvement_vs_exp12:.1f}pp")
else:
    gap = (0.90 - stack_wf1_final) * 100
    print(f"  Gap from 90%: {gap:.1f}pp")
    print(f"  Consider: More Optuna trials, feature engineering, or threshold tuning")

print("=" * 70)